# VLM Comparison Experiment

Bu notebook iki VLM deneyi yapar:

1. Image-only VLM: modele sadece cephe gorseli verilir.
2. Detection-assisted VLM: ayni gorsel + YOLO/SAM2 tespit JSON'u verilir.

Amac: VLM tek basina segmentasyon/raporlamada ne kadar basarili, bizim sistemin ciktisi verilince rapor kalitesi nasil degisiyor?

## 1. Repo ve Paketler

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt
!pip install -q gdown

## 2. VLM Ayarlari

OpenAI-compatible bir endpoint kullanir. OpenAI API icin `VLM_BASE_URL = 'https://api.openai.com/v1'` kalabilir. Local/vLLM server kullanacaksan base URL'i kendi endpoint'inle degistir.

In [ ]:
import os
from getpass import getpass

VLM_BASE_URL = 'https://api.openai.com/v1'
VLM_MODEL = 'MODEL_ADINI_BURAYA_YAZ'  # Ornek: kullandigin vision model adi
VLM_API_KEY = os.environ.get('VLM_API_KEY') or getpass('VLM API key: ')

assert VLM_MODEL != 'MODEL_ADINI_BURAYA_YAZ', 'VLM_MODEL degerini kullandigin model adi ile degistir.'

## 3. Test Gorsellerini ve Hibrit Sonuclari Al

Eger daha once `hybrid_yolo_sam2_colab.ipynb` calistirdiysan, Drive reports klasorundeki son hybrid klasorunu zip olarak yukleyebilir veya path'i verebilirsin. Burada varsayilan olarak manuel upload desteklenir.

In [ ]:
from google.colab import files
import zipfile, shutil
from pathlib import Path

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
WORK_DIR = PROJECT_DIR / 'outputs/vlm_comparison_input'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    path = WORK_DIR / name
    path.write_bytes(content)
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as zf:
            zf.extractall(WORK_DIR)

print('Input files:')
for path in sorted(WORK_DIR.rglob('*')):
    print(path)

## 4. Gorsel ve JSON Sec

Bir test gorseli secilir. Notebook ayni gorselin YOLO raw, filtered ve SAM2 hybrid kayitlarini bulmaya calisir.

In [ ]:
from pathlib import Path
import json

image_candidates = [p for p in WORK_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif'}]
json_candidates = {p.name: p for p in WORK_DIR.rglob('*.json')}
print('Images:')
for i, p in enumerate(image_candidates):
    print(i, p)
print('JSON files:', list(json_candidates))

IMAGE_INDEX = 0
image_path = image_candidates[IMAGE_INDEX]
print('Selected image:', image_path)

raw_json = json_candidates.get('yolo_raw_conf025.json')
filtered_json = json_candidates.get('yolo_filtered_conf070.json')
hybrid_json = json_candidates.get('hybrid_yolo_conf070_sam2.json')
print('raw:', raw_json)
print('filtered:', filtered_json)
print('hybrid:', hybrid_json)

## 5. Yardimci Fonksiyonlar

In [ ]:
import base64
import json
import mimetypes
import requests
from pathlib import Path
from PIL import Image as PILImage


def prepare_image_for_vlm(path: Path) -> tuple[Path, str]:
    # Some APIs/IPython paths do not support AVIF well. Convert every input to JPEG for stable VLM upload.
    out_dir = PROJECT_DIR / 'outputs/vlm_comparison'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{path.stem}_vlm.jpg'
    image = PILImage.open(path).convert('RGB')
    image.save(out_path, quality=92)
    return out_path, 'image/jpeg'


def image_data_url(path: Path) -> str:
    prepared, mime = prepare_image_for_vlm(path)
    encoded = base64.b64encode(prepared.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'


def load_records(path: Path | None):
    if path is None:
        return []
    data = json.loads(path.read_text())
    return data.get('records', data) if isinstance(data, dict) else data


def image_name_from_record(record):
    return Path(str(record.get('image', record.get('path', '')))).name


def record_for_image(path: Path, json_path: Path | None):
    target = path.name
    records = load_records(json_path)
    for record in records:
        if image_name_from_record(record) == target or image_name_from_record(record).endswith(target):
            return record
    # fallback for refined visuals: sam2_refined__t1.jpg vs t1.jpg
    target_stem = path.stem.replace('sam2_refined__', '')
    for record in records:
        if Path(str(record.get('image', ''))).stem == target_stem:
            return record
    return None


def compact_detection_record(record, max_detections=80):
    if not record:
        return None
    detections = record.get('detections', [])[:max_detections]
    compact = {
        'image': record.get('image'),
        'width': record.get('width'),
        'height': record.get('height'),
        'detection_count': len(record.get('detections', [])),
        'detections': []
    }
    for det in detections:
        compact['detections'].append({
            'label': det.get('label'),
            'confidence': det.get('confidence'),
            'bbox_xyxy': det.get('bbox_xyxy'),
            'sam2_score': det.get('sam2_score'),
            'bbox_area_ratio': det.get('bbox_area_ratio'),
        })
    return compact


def call_vlm(messages, temperature=0.2):
    response = requests.post(
        f"{VLM_BASE_URL.rstrip('/')}/chat/completions",
        headers={'Authorization': f'Bearer {VLM_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': VLM_MODEL, 'messages': messages, 'temperature': temperature},
        timeout=180,
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content']

SYSTEM_PROMPT = '''Turkce yanit veren, mimari cephe lejant raporu hazirlayan teknik bir asistansin.
Belirsizligi acikca belirt. Sayilari uydurma. Segmentasyon yapabiliyorsan sinirlarini tarif et; piksel/poligon veremiyorsan bunu soyle.'''


## 6A. Deney A - Sadece Gorsel ile VLM

In [ ]:
image_url = image_data_url(image_path)
image_only_prompt = '''Bu cephe gorselini incele.

Istenen cikti:
1. Cephede gordugun mimari elemanlari listele.
2. Mümkünse adet tahmini yap, ama emin degilsen tahmin oldugunu belirt.
3. Segmentasyon yapabiliyorsan hangi elemanlarin nerede oldugunu tarif et.
4. Mimari cephe lejant raporu gibi kisa ve denetlenebilir yaz.'''

image_only_report = call_vlm([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': [
        {'type': 'text', 'text': image_only_prompt},
        {'type': 'image_url', 'image_url': {'url': image_url}},
    ]},
])
print(image_only_report)

## 6B. Deney B - Bizim Sistem Ciktisi + Gorsel ile VLM

In [ ]:
raw_record = compact_detection_record(record_for_image(image_path, raw_json), max_detections=40)
filtered_record = compact_detection_record(record_for_image(image_path, filtered_json), max_detections=80)
hybrid_record = compact_detection_record(record_for_image(image_path, hybrid_json), max_detections=80)

payload = {
    'raw_yolo_conf025': raw_record,
    'filtered_yolo_conf070': filtered_record,
    'hybrid_yolo_sam2': hybrid_record,
}

assisted_prompt = '''Ayni cephe gorseli ve YOLO/SAM2 sistem ciktisi asagidadir.

Gorevin:
1. JSON'daki tespitleri kullanarak mimari cephe lejant raporu yaz.
2. Raw YOLO ile filtered/hybrid sonucu karsilastir.
3. Fazla tekrar, supheli sinif ve eksik tespitleri belirt.
4. VLM olarak kendi gorsel yorumun ile JSON arasinda celiski varsa ayri baslikta yaz.
5. Sayilari sadece JSON'dan al; emin olmadigin gorsel yorumlari sayisal kesinlik gibi yazma.

JSON:
'''

assisted_report = call_vlm([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': [
        {'type': 'text', 'text': assisted_prompt + json.dumps(payload, ensure_ascii=False, indent=2)},
        {'type': 'image_url', 'image_url': {'url': image_url}},
    ]},
])
print(assisted_report)

## 6C. Deney C - Iki Cevabi Karsilastir

In [ ]:
comparison_prompt = f'''Asagida ayni cephe gorseli icin iki VLM raporu var.

A) Sadece gorsel raporu:
{image_only_report}

B) YOLO/SAM2 destekli rapor:
{assisted_report}

Bu iki sonucu bilimsel deney diliyle karsilastir:
- Hangisi daha denetlenebilir?
- Hangisi sayisal olarak daha guvenilir?
- VLM tek basina segmentasyon yapabildi mi?
- Sistem destekli VLM rapor kalitesini artirdi mi?
- Makaleye yazilacak kisa sonuc paragrafi oner.
'''

comparison_report = call_vlm([
    {'role': 'system', 'content': 'Turkce yazan, bilgisayarla gorme deneylerini raporlayan akademik bir asistansin.'},
    {'role': 'user', 'content': comparison_prompt},
])
print(comparison_report)

## 7. Raporlari Kaydet

In [ ]:
from pathlib import Path
import json

out_dir = PROJECT_DIR / 'outputs/vlm_comparison'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'image_only_report.md').write_text(image_only_report, encoding='utf-8')
(out_dir / 'detection_assisted_report.md').write_text(assisted_report, encoding='utf-8')
(out_dir / 'comparison_report.md').write_text(comparison_report, encoding='utf-8')
(out_dir / 'payload.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved to:', out_dir)
!find /content/lejanter_doga_vlm_codex/outputs/vlm_comparison -maxdepth 1 -type f | sort